# Raw - Extract StatCan data via API - FDI, Imports, Exports

**GAC's data rules for Foreign Direct Investment (FDI) (also in repo `README.md`)**
-   Stocks vs. Flows
    -   **Use stocks by default**.
        -   _Stocks_ = cumulative investment up to year-end.
        -   _Flows_ = investment during a single year (see Table 36-10-0473-01).
-   Currency Conversion
    -   Convert foreign-currency FDI stocks to CAD using the year-end exchange rate (point-in-time measure).
-   FDI in Canada
    -   Prefer **ultimate investor country** (Table 36-10-0433-01).
    -   If unavailable, use **immediate investor country** (Table 36-10-0008-01).
-   Canadian Direct Investment Abroad (CDIA)
    -   Always report by **immediate investor country** (Table 36-10-0008-01).
    -   CDIA by ultimate destination is **not available**.

### FDI types and their respective StatCan table id:

| FDI Type | Investor Basis | StatCan Table ID | Table ID for StatCan API |
| :--- | :--- | :--- | :--- |
**FDI in Canada (Preferred)** | Ultimate | 36-10-0433-01 | 36100433
**FDI in Canada (Alternative)**| Immediate | 36-10-0008-01 | 36100008
**FDI from Canada, Outbound (CDIA)** | Immediate | 36-10-0008-01 | 36100008
**FDI Flows (If needed)** | N/A | 36-10-0473-01 | 36100473

### Trade types and their respective Statcan table id:

| Trade Type | Trade Direction | StatCan Table ID | Table ID for StatCan API |
| :--- | :--- | :--- | :--- |
**Imports** | FROM foreign regions TO Canada | 12-10-0115-01 | 12100115
**Exports**| FROM Canada TO foreign regions | 12-10-0104-01 | 12100104


## 1. Setup

In [2]:
from time import sleep
from pathlib import Path 
import pandas as pd 
import sys

project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

from utils.etl_utils import (
    get_full_table_zip_url,
    download_zip,
    extract_zip_file
)
from utils.paths import RAW_STATCAN_DIR

### 1.1 Set tables to be requested

In [3]:
FDI_IMMEDIATE_TABLE_ID  = 36100008
FDI_ULTIMATE_TABLE_ID   = 36100433
IMPORTS_TABLE_ID = 12100115
EXPORTS_TABLE_ID = 12100104

tables_to_csv = [
    FDI_IMMEDIATE_TABLE_ID,
    FDI_ULTIMATE_TABLE_ID,
    IMPORTS_TABLE_ID,
    EXPORTS_TABLE_ID
]

## 2. Make API calls to StatCan

Use `getFullTableDownloadCSV` API method

In [7]:
for table_id in tables_to_csv:
    zip_filepath = RAW_STATCAN_DIR / f"raw_statcan__tbl_{table_id}.zip"
    zip_url = get_full_table_zip_url(table_id=table_id, verbose=True)

    # FIXED: use `destination` instead of `dest`
    download_zip(
        url=zip_url,
        destination=zip_filepath,
        verbose=True,   # `chunk` will just use the default 8192
    )

    extract_zip_file(
        zip_path=zip_filepath,
        out_file_prefix="raw_statcan__tbl_"
    )

    print()
    sleep(1)


GET URL status: HTTP 200: OK
Download status: HTTP 200: OK
Extracting raw_statcan__tbl_36100008.zip... Done!

GET URL status: HTTP 200: OK
Download status: HTTP 200: OK
Extracting raw_statcan__tbl_36100433.zip... Done!

GET URL status: HTTP 200: OK
Download status: HTTP 200: OK
Extracting raw_statcan__tbl_12100115.zip... Done!

GET URL status: HTTP 200: OK
Download status: HTTP 200: OK
Extracting raw_statcan__tbl_12100104.zip... Done!



### 2.3 Notes on StatCan data

From bottom `note` section of metadata tables (`*_MetaData.csv`).

#### 2.3.1 Table 36100008: International investment position, Canadian direct investment abroad and foreign direct investment in Canada, by country, annual

##### Data source & subject

Survey Name: Canada's International Investment Position
Survey Code: 1537

Subject Name: Economic accounts
Subject Code: 36

##### Notes

1. This is the sum of all countries, including data for countries not listed in this table.

2. Starting in 2019, Antigua and Barbuda, Aruba, Bahamas, Barbados, Bermuda, British Virgin Islands, Cayman Islands, Cuba, Dominica, Dominican Republic, Grenada, Guadeloupe, Haiti, Jamaica, Saint Lucia and Trinidad and Tobago, are classified to the region "Caribbean" instead of "North America". The region "North America" includes the "United States", "Mexico" and "Canada" when applicable. The data for the "North America" region have been updated starting with 1987.

3. Users are cautioned that, prior to 2019, data for smaller countries (generally defined as countries with foreign direct investment below 500 million dollars) is subject to higher sampling variability.

4. Swaziland was renamed Eswatini in 2018.

5. Starting in 2019, Turkmenistan is classified to "Asia/Oceania" instead of "Europe."

6. "Unallocated countries" include all stakes that could not be allocated to a particular country.

7. Members that include "Other" display the value of the other countries comprised in the region but not displayed in the table.

8. Starting in 2019, data for the United States no longer include Puerto Rico and the United States Virgin Islands. These are now included in “Other Caribbean.”

---

#### 2.3.2 Table 12100115: Trade in goods by importer characteristics, by country of export

##### Data source & subject

Survey Name: Trade by Importer Characteristics - Goods
Survey Code: 5237

Subject Name: International trade
Subject Code: 12

##### Notes

1. The country of export is the country from which the goods were exported into Canada, as reported on the import declaration form. It can differ from the country of origin. For additional information and definitions related to the country of export variable, refer to  <a href="https://www150.statcan.gc.ca/n1/pub/71-607-x/2021004/concepts-eng.htm" rel="external noopener noreferrer" target="_blank">Canadian International Merchandise Trade Database Concepts</a>.

2. Data may not add to totals due to rounding.

3. Territories include: Yukon, Northwest Territories and Nunavut.

4. An importing establishment is included in all different countries it imported from during the reference year. For example, an establishment that imported from the United States and China is counted once for the United States and once for China. As such, the number of importing establishments by country of export are non-additive.

5. Total value of imports refers to the part of annual import value (customs basis) that can be linked to specific entities in the Business Register each year. Annual import values (customs basis) can be obtained from table 12-10-0175. For additional information and definitions related to merchandise imports, refer to <a href="http://www23.statcan.gc.ca/imdb/p2SV.pl?Function=getSurvey&SDDS=2201" rel="external noopener noreferrer" target="_blank"> Canadian International Merchandise Trade (Custom Basis)</a>.

---

#### 2.3.3 Table 12100104: Trade in goods by exporter characteristics, by country of destination

##### Data source & subject

Survey Name: Trade by Exporter Characteristics - Goods
Survey Code: 5124

Subject Name: International trade
Subject Code: 12

##### Notes

1. Data may not add to totals due to rounding.

2. Territories include: Yukon, Northwest Territories and Nunavut.

3. An exporting establishment is included in all different countries it exported to during the reference year. For example, an establishment that exported to the United States and China is counted once for the United States and once for China. As such, the number of exporting establishments by country of destination are non-additive.

4. Total value of exports refers to the part of annual domestic export value (customs basis) that can be linked to specific entities in the Business Register each year. Annual domestic export values (customs basis) can be obtained from table 12-10-0175. <br> For additional information and definitions related to merchandise domestic exports, refer to <a href="http://www23.statcan.gc.ca/imdb/p2SV.pl?Function=getSurvey&SDDS=2201" rel="external noopener noreferrer" target="_blank"> Canadian International Merchandise Trade (Custom Basis)</a>.